# Day-ahead consumption forecast (fixed)

The mock is mostly right. Two real changes, plus notes on the things that looked wrong but were not.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.width", 120)
df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time")
fc = pd.read_csv("../../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
s = df.set_index("time")["consumption_mwh"]
frame = pd.DataFrame({"y": s})
frame["decision_time"] = frame.index.normalize() - pd.Timedelta(days=1) + pd.Timedelta(hours=12)
frame["lag48"] = s.shift(48); frame["lag168"] = s.shift(168); frame["lag336"] = s.shift(336)
frame["roll24_at_decision"] = s.shift(1).rolling(24).mean().reindex(frame["decision_time"]).to_numpy()
tgt = frame.reset_index().rename(columns={"time": "target_time"})[["target_time", "decision_time"]].sort_values("decision_time")
joined = pd.merge_asof(tgt, fc.sort_values("origin_datetime"), left_on="decision_time", right_on="origin_datetime",
                       direction="backward", left_by="target_time", right_by="forecast_datetime").set_index("target_time").sort_index()
frame["temp_fc"] = joined["temp_forecast_c"].reindex(frame.index).to_numpy()
frame["hdd"] = np.clip(15 - frame["temp_fc"], 0, None); frame["cdd"] = np.clip(frame["temp_fc"] - 22, 0, None)
frame["is_weekend"] = (frame.index.dayofweek >= 5).astype(float)
hour_dummies = pd.get_dummies(frame.index.hour, prefix="h", drop_first=True).astype(float); hour_dummies.index = frame.index
frame = pd.concat([frame, hour_dummies], axis=1)

*Fix 1 — drop the exactly collinear feature.* `diff_week = lag168 - lag336` is a linear
combination of two features already in the model. Predictions are unaffected, but Ridge
splits the weight arbitrarily (the mock printed lag168 291, lag336 297, diff_week -15), so
the coefficient table cannot be read. Remove it.

In [2]:
features = ["lag48", "lag168", "lag336", "roll24_at_decision", "temp_fc", "hdd", "cdd", "is_weekend"] + list(hour_dummies.columns)
data = frame.dropna(subset=features + ["y"])
split_time = pd.Timestamp("2023-08-01", tz="UTC")
train, test = data[data.index < split_time], data[data.index >= split_time]
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(train[features], train["y"])
pred = pd.Series(model.predict(test[features]), index=test.index)
coef = pd.Series(model.named_steps["ridge"].coef_, index=features)
print(coef.drop(hour_dummies.columns).sort_values().round(0))

is_weekend            -868.0
temp_fc               -126.0
lag48                  -18.0
cdd                     28.0
roll24_at_decision     235.0
lag168                 250.0
lag336                 337.0
hdd                   1548.0
dtype: float64


*Fix 2 — the naive baseline must respect the same decision time.* "Same hour yesterday"
(`shift(24)`) uses D-1 values for hours 12:00-23:00 that are not observed at 12:00 on D-1.
The honest persistence baseline is D-1 for hours before 12:00 and D-2 otherwise (or simply
D-2 for everything). The model's advantage over the honest baseline is larger, not smaller,
but the point is that baseline and model must be allowed the same information.

In [3]:
def metrics(y, p): return {"RMSE": np.sqrt(mean_squared_error(y, p)), "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p)}
hour = s.index.hour
persist = pd.Series(np.where(hour < 12, s.shift(24), s.shift(48)), index=s.index).reindex(test.index)
results = {
    "shift(24) as in the mock (uses unobserved hours)": metrics(test["y"], s.shift(24).reindex(test.index)),
    "honest persistence (D-1 before 12:00, else D-2)": metrics(test["y"], persist),
    "same hour last week": metrics(test["y"], test["lag168"]),
    "ridge (no diff_week)": metrics(test["y"], pred),
}
out = pd.DataFrame(results).T
out.round(3)

,RMSE,MAE,R2
shift(24) as in the mock (uses unobserved hours),1617.828,1272.509,0.833
"honest persistence (D-1 before 12:00, else D-2)",1776.903,1418.581,0.798
same hour last week,1405.158,1099.668,0.874
ridge (no diff_week),898.677,710.705,0.948


## Red herrings — things that look suspicious and are fine

- `Ridge(random_state=0)`: only used by stochastic solvers (sag/saga); harmless with the default.
- `s.shift(1).rolling(24).mean()` re-indexed at the decision time: the window ends at 11:00 D-1, all observed by 12:00.
- `StandardScaler` inside the pipeline: fitted on train only when `fit` is called on the pipeline.
- `get_dummies(..., drop_first=True)` with a model that has an intercept: correct; keeping all 24 would be the mistake.
- `merge_asof(direction="backward")` on `decision_time` vs `origin_datetime` with `left_by/right_by` on the target hour, plus the assert: this is the honest point-in-time join (horizons 12-35 h).
- `ffill()` on `temp_fc`: only ever fills from earlier rows; here it is a no-op except the first archive day, which is dropped.
- Single `dropna` on the combined frame: keeps X and y aligned.
- No gap between train and test: the target sits at the row's own time and every feature is lagged, so no train target overlaps a test feature.
- lag48 rather than lag24: correct given the 12:00 D-1 decision time.
- R² 0.948 with RMSE ~900 MWh: consistent with the naive baseline's 0.83 and with hourly demand of ~29 GWh; not suspicious.
- UTC hour dummies: consistent throughout; local-time dummies would be a refinement, not a fix.

In [4]:
print(out.round(3).to_string())
print(f"\nRidge RMSE {out.iloc[3, 0]:,.0f} vs honest persistence {out.iloc[1, 0]:,.0f} MWh ({1 - out.iloc[3, 0] / out.iloc[1, 0]:.0%} improvement); R2 {out.iloc[3, 2]:.3f}.")

                                                      RMSE       MAE     R2
shift(24) as in the mock (uses unobserved hours)  1617.828  1272.509  0.833
honest persistence (D-1 before 12:00, else D-2)   1776.903  1418.581  0.798
same hour last week                               1405.158  1099.668  0.874
ridge (no diff_week)                               898.677   710.705  0.948

Ridge RMSE 899 vs honest persistence 1,777 MWh (49% improvement); R2 0.948.
